In [ ]:
# ruff: noqa: N803, N806
from typing import Callable

import geopandas as gpd
import hvplot.xarray  # noqa: F401
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.optimize
import scipy.spatial
import xarray as xr
from numpy.typing import ArrayLike
from rich.progress import track
from scipy.optimize import curve_fit

# Import cupy (cuda accelerated numpy) if available
try:
    import cupy as cp
    import cupy_xarray  # noqa: F401
    import cupyx.scipy.spatial
except ImportError:
    cp = None

In [ ]:
import time
from collections import defaultdict
from contextlib import contextmanager
from statistics import mean, stdev


class StopUhr:
    def __init__(self):
        self.reset()

    def reset(self):
        self.durations = defaultdict(list)
        self.resolutions = defaultdict(int)

    def result(self, msg: str):
        res = self.resolutions[msg]
        if len(self.durations[msg]) == 0:
            print(f"'{msg}' was not measured")
        elif len(self.durations[msg]) == 1:
            print(f"'{msg}' took {self.durations[msg][0]}s")
        else:
            total_duration = round(sum(self.durations[msg]), res)
            avg_duration = round(mean(self.durations[msg]), res)
            std_duration = round(stdev(self.durations[msg]), res)
            print(f"'{msg}' took {total_duration} in total with an average of {avg_duration}s +- {std_duration}s")

    def results(self):
        for msg in self.durations.keys():
            self.result(msg)

    @contextmanager
    def __call__(self, msg: str, res: int = 2, loop: bool = False):
        """Measure the time it takes to execute a block of code.

        Args:
            msg (str): A unique message to identify the block of code.
            res (int, optional): The number of decimal places to round the time to. Defaults to 2.

        Yields:
            float: The start time of the block of code.

        """
        assert isinstance(msg, str), "msg must be a string"
        assert isinstance(res, int) and res >= 0, "res must be a non-negative integer"
        use_ns = res > 6
        tick_start = time.perf_counter_ns() if use_ns else time.perf_counter()
        yield tick_start
        tick_end = time.perf_counter_ns() if use_ns else time.perf_counter()
        duration = round(tick_end - tick_start, res)
        if loop:
            self.durations[msg].append(duration)
            self.resolutions[msg] = res
        else:
            print(f"'{msg}' took {duration}s")


stopuhr = StopUhr()

## Load and visualize the data

In [ ]:
waterquality = pd.read_csv(
    "data/NationalSurveyData.csv",
    sep=",",
    skipinitialspace=True,
    skiprows=(0, 1, 2, 3, 5, 6),
)
# We are only interest in the As column (and lat+lon)
waterquality = waterquality[["SAMPLE_ID", "As", "LAT_DEG", "LONG_DEG"]]
# Convert the Aq values to numeric
waterquality["As"] = waterquality["As"].str.replace(r"^<", "", regex=True).astype(float)
# Convert to a GeoDataFrame
waterquality = gpd.GeoDataFrame(
    waterquality,
    geometry=gpd.points_from_xy(waterquality["LONG_DEG"], waterquality["LAT_DEG"]),
).set_crs(epsg=4326)
# Normalize the geometries and drop duplicate points
waterquality["geometry"] = waterquality.normalize()
waterquality = waterquality.drop_duplicates(subset="geometry")
# Convert to a better fitting crs
waterquality = waterquality.to_crs(epsg=9678)
display(waterquality)
surveyarea = gpd.read_file("data/borders.shp")
surveyarea = surveyarea.to_crs(epsg=9678)
display(surveyarea)

# Plot the survey area
vmax = np.percentile(waterquality["As"], 95)
m = waterquality.explore(column="As", cmap="PiYG_r", tiles="CartoDB positron", vmax=vmax)
surveyarea.boundary.explore(m=m, style_kwds={"fill": False, "color": "black"})
m

## Emprical indicator (Semi-)Variogram

In [ ]:
def fastdist(a: gpd.GeoSeries, b: gpd.GeoSeries, to_host: bool = False) -> xr.DataArray:
    """Calculate the pairwise distances between two sets of geometries.

    Utilizes cupy if available for GPU acceleration.

    Args:
        a (gpd.GeoSeries): The first set of geometries.
        b (gpd.GeoSeries): The second set of geometries.
        to_host (bool, optional): Whether to return the result on the host. Defaults to False.

    Returns:
        xr.DataArray: The pairwise distances between the two sets of geometries.

    """
    if cp is not None:
        a_arr = cp.array([a.x, a.y], order="F").T.reshape(-1, 2)
        b_arr = cp.array([b.x, b.y], order="F").T.reshape(-1, 2)
        d = cupyx.scipy.spatial.distance_matrix(a_arr, b_arr)
        if to_host:
            d = d.get()
    else:
        a_arr = np.array([a.x, a.y], order="F").T.reshape(-1, 2)
        b_arr = np.array([b.x, b.y], order="F").T.reshape(-1, 2)
        d = scipy.spatial.distance_matrix(a_arr, b_arr)
    return xr.DataArray(d, dims=("a", "b"), coords={"a": a.index, "b": b.index})

In [ ]:
def empirical_semivariogram(z: gpd.GeoSeries, geom: gpd.GeoSeries, h: ArrayLike | None = None) -> xr.DataArray:
    """Calculate the empirical semivariogram.

    Args:
        z (gpd.GeoSeries): The data to calculate the semivariogram for.
        geom (gpd.GeoSeries): The geometries of the data. Must be points.
        h (list[float] | None): The bins to calculate the semivariogram for.
            If None, the bins are estimated by linspacing 0 - distances.median() with 40 bins total.
            Defaults to None.

    Returns:
        xr.DataArray: The empirical semivariogram.

    """
    # Calculate the pairwise distances
    d = fastdist(geom, geom, to_host=True).rename({"a": "tail", "b": "head"})
    # Remove upper triangle
    d = d.where(np.triu(np.ones(d.shape), k=1).astype(bool))
    # Remove self-distances
    d = d.where(~np.eye(len(z), dtype=bool))

    # If h is not provided, estimate the lags manually
    if h is None:
        h = np.linspace(0, d.median(), 40)

    # Calculate the pairwise data differences
    z_head = xr.DataArray(z, coords={"head": z.index}, name="As").expand_dims({"tail": z.index})
    z_tail = xr.DataArray(z, coords={"tail": z.index}, name="As").expand_dims({"head": z.index})
    z_diff = z_head - z_tail

    # Calculate the semivariogram
    gamma = xr.DataArray(0.0, dims=["lag"], coords={"lag": h[1:]})
    for i in range(1, len(h)):
        is_in_lag = (h[i - 1] < d) & (d <= h[i])
        n_h = is_in_lag.sum(dim=["tail", "head"])
        gamma.loc[{"lag": h[i]}] = 1 / (2 * n_h) * (z_diff.where(is_in_lag) ** 2).sum(["tail", "head"])

    # This vectorized approach is taken too much memory, but could potentially be faster
    # d_lags = xr.DataArray(np.digitize(d, h, right=True), coords={"tail": z.index, "head": z.index})
    # is_in_lag = xr.DataArray(
    #   [d_lags == i for i in range(1, len(h))],
    #   coords={"lag": h[1:], "tail": z.index, "head": z.index}
    # )
    # n_h = is_in_lag.sum(dim=["tail", "head"])
    # gamma = 1 / (2 * n_h) * (z_diff.where(is_in_lag) ** 2).sum(["tail", "head"])

    return gamma


gamma = empirical_semivariogram((waterquality["As"] >= 10).astype(int), waterquality.geometry)
gamma

## Fit a theoretical indicator (Semi-)Variogram

In [ ]:
def spherical(h: ArrayLike, c0: float, c: float, a: float) -> ArrayLike:
    """Calculate the theoretical semivariogram from a spherical model.

    Args:
        h (ArrayLike): The lags to calculate the semivariogram for.
        c0 (float): The nugget effect.
        c (float): The sill.
        a (float): The range.

    Returns:
        ArrayLike: The spherical semivariogram model.

    """
    semivar = np.zeros_like(h)
    curve_mask = (0 < h) & (h <= a)
    line_mask = h > a
    semivar[curve_mask] = c0 + c * (3 * h[curve_mask] / (2 * a) - 0.5 * (h[curve_mask] / a) ** 3)
    semivar[line_mask] = c0 + c
    return semivar


def spherical_fast(h: ArrayLike, c0: float, c: float, a: float) -> ArrayLike:
    """Calculate the theoretical semivariogram from a spherical model without satisfying constraints.

    Should not be used for fitting!
    Should not be used for h larger than a!

    Args:
        h (ArrayLike): The lags to calculate the semivariogram for.
        c0 (float): The nugget effect.
        c (float): The sill.
        a (float): The range.

    Returns:
        ArrayLike: The spherical semivariogram model.

    """
    return c0 + c * (3 * h / (2 * a) - 0.5 * (h / a) ** 3)


def cov_from_semivar(semivar: ArrayLike, c0: float, c: float) -> ArrayLike:
    """Calculate the covariance from a semivariogram model.

    Args:
        semivar (ArrayLike): The semivariogram model.
        c0 (float): The nugget effect.
        c (float): The sill.

    Returns:
        ArrayLike: The covariance model.

    """
    return c + c0 - semivar


(c0, c, a), cov = curve_fit(spherical, gamma.coords["lag"], gamma, p0=[0, gamma.mean(), gamma.coords["lag"].mean()])
c0, c, a

In [ ]:
fig, ax = plt.subplots()
gamma.plot(marker="o", ax=ax, label="Empirical Semi-Variogram")
ax.plot(
    gamma.coords["lag"], spherical(gamma.coords["lag"], c0, c, a), label="Theoretical Semi-Variogram (Spherical Model)"
)
ax.plot(
    gamma.coords["lag"],
    spherical_fast(gamma.coords["lag"], c0, c, a),
    linestyle=":",
    label="Theoretical Semi-Variogram (Fast Spherical Model)",
)
ax.plot(
    gamma.coords["lag"],
    cov_from_semivar(spherical(gamma.coords["lag"], c0, c, a), c0, c),
    label="Covariance (Spherical Model)",
)
# Plot horizontal line at sill
ax.axhline(c + c0, color="black", linestyle="--", label="Sill + Nugget")
# Plot vertical line at range
ax.axvline(a, color="grey", linestyle="--", label="Range")
ax.set_xlabel("Lag Distance")
ax.set_ylabel("Semivariance / Covariance")
ax.legend()
plt.show()

## Indicator Kriging

In [ ]:
# Create a grid, spanning the area of interest
n_points = 100
xy = np.meshgrid(
    np.linspace(surveyarea.bounds.minx.item(), surveyarea.bounds.maxx.item(), n_points),
    np.linspace(surveyarea.bounds.miny.item(), surveyarea.bounds.maxy.item(), n_points),
)
grid_gdf = gpd.GeoDataFrame(geometry=gpd.points_from_xy(xy[0].ravel(), xy[1].ravel())).set_crs(surveyarea.crs)
# Filter out points that are not within the survey area
grid_gdf = grid_gdf[grid_gdf.within(surveyarea.loc[0, "geometry"])]
len(grid_gdf)

In [ ]:
# Create a grid, spanning the area of interest
res = 2000  # m
x = np.arange(surveyarea.bounds.minx.item(), surveyarea.bounds.maxx.item(), res)
y = np.arange(surveyarea.bounds.miny.item(), surveyarea.bounds.maxy.item(), res)
X, Y = np.meshgrid(x, y)
print(X.shape, Y.shape)
grid_gdf = gpd.GeoDataFrame(geometry=gpd.points_from_xy(X.ravel(), Y.ravel(), crs=surveyarea.crs))
# Filter out points that are not within the survey area
grid_gdf = grid_gdf[grid_gdf.within(surveyarea.loc[0, "geometry"])]
len(grid_gdf)

In [ ]:
# ruff: noqa: N803, N806
def ordinary_kriging(
    u: gpd.GeoSeries,
    z: gpd.GeoSeries,
    geom: gpd.GeoSeries,
    model_fn: Callable,
    c0: float,
    c: float,
    a: float,
    filter_range: bool = True,
) -> xr.DataArray:
    """Interpolate points u with ordinary kriging based on a fitted semi-variogram model and support points z.

    Args:
        u (gpd.GeoSeries): The points to interpolate ("nodes").
        z (gpd.GeoSeries): The values of the support points.
        geom (gpd.GeoSeries): The geometries of the support points.
        model_fn (Callable): The model function to use for the semi-variogram.
        c0 (float): The nugget effect.
        c (float): The sill.
        a (float): The range.
        filter_range (bool, optional): Whether to filter out distances larger than the range. Defaults to True.

    Returns:
        xr.DataArray: The interpolated values.

    """
    with stopuhr("Caculating C(0)"):
        # Calculate the covariance matrix C(0) between the unknown points (nodes) and the support points
        d0 = fastdist(u, geom)
        C0 = cov_from_semivar(model_fn(d0, c0, c, a), c0, c)
        # Filter out distances which are larger than the range a, since we don't want to use them
        if filter_range:
            C0 = C0.where(d0 <= a, 0)
        # Rename the dimensions to match our naming scheme for the rest of the function
        C0 = C0.rename({"a": "nodes", "b": "support"})

    with stopuhr("Caculating C(h)"):
        # Calculate the pariwise covariance matrix C(h) between the support points
        dh = fastdist(geom, geom)
        Ch = cov_from_semivar(model_fn(dh, c0, c, a), c0, c).rename({"a": "head", "b": "tail"})

    with stopuhr("Constraining C(0) and C(h)"):
        # Apply the constraints that the lambdas must sum up to 1
        C0_constrained = C0.pad({"support": (0, 1)}, constant_values=1)
        Ch_constrained = Ch.pad({"tail": (0, 1), "head": (0, 1)}, constant_values=1)
        Ch_constrained[{"tail": -1, "head": -1}] = 0

    with stopuhr("Calculating lambda"):
        # Calculate the weights on cuda if available, else via numpy
        if cp is not None:
            C0_constrained = C0_constrained.cupy.as_cupy()
            Ch_constrained = Ch_constrained.cupy.as_cupy()
            lam = cp.linalg.solve(
                Ch_constrained.data,
                C0_constrained.transpose("support", "nodes").data,
            )
        else:
            lam = np.linalg.solve(
                Ch_constrained.data,
                C0_constrained.transpose("support", "nodes").data,
            )
        lam = xr.DataArray(
            lam,
            coords=C0_constrained.transpose("support", "nodes").coords,
            name="lambda",
        )
        lam, mu = lam[:-1], lam[-1]
        mu = mu.rename("mu")

    z = xr.DataArray(z, coords={"support": z.index})
    if cp is not None:
        z = z.cupy.as_cupy()
    with stopuhr("Calculating kriged"):
        # Calculate the kriged values
        kriged = xr.dot(lam, z, dims="support").rename("kriged")

    if cp is not None:
        kriged = kriged.cupy.as_numpy()
        mu = mu.cupy.as_numpy()
        cp.get_default_memory_pool().free_all_blocks()

    return kriged, mu


with stopuhr("Kriging"):
    k = 10
    z = waterquality["As"]
    z_encoded = (z >= k).astype(int)
    kriged_grid, mu_grid = ordinary_kriging(
        grid_gdf.geometry, z_encoded, waterquality.geometry, spherical_fast, c0, c, a
    )
    grid_gdf[f"kriged_{k}"] = kriged_grid.to_numpy()
    grid_gdf[f"mu_{k}"] = mu_grid.to_numpy()

# Visualize the grid
vmax = np.percentile(waterquality["As"], 95)
m = waterquality.explore(column="As", cmap="PiYG_r", tiles="CartoDB positron", vmax=vmax)
grid_gdf.explore(m=m, column="kriged_10", marker_kwds={"radius": 3, "fill": True})
surveyarea.boundary.explore(m=m, style_kwds={"fill": False, "color": "black"})
m

In [ ]:
data_variance = (waterquality["As"] >= 10).astype(int).std()
kriged_variance = grid_gdf["kriged_10"].std()
print(f"Data Variance: {data_variance:.2f}")
print(f"Kriged Variance: {kriged_variance:.2f}")
assert data_variance > kriged_variance, "The kriged variance is larger than the data variance, which is not plausible."

In [ ]:
del kriged_grid, mu_grid
cp.get_default_memory_pool().free_all_blocks()

## Indicator Simulation

In [ ]:
# Create a grid, spanning the area of interest
n_points = 100
xy = np.meshgrid(
    np.linspace(surveyarea.bounds.minx.item(), surveyarea.bounds.maxx.item(), n_points),
    np.linspace(surveyarea.bounds.miny.item(), surveyarea.bounds.maxy.item(), n_points),
)
grid_gdf = gpd.GeoDataFrame(geometry=gpd.points_from_xy(xy[0].ravel(), xy[1].ravel())).set_crs(surveyarea.crs)
# Filter out points that are not within the survey area
grid_gdf = grid_gdf[grid_gdf.within(surveyarea.loc[0, "geometry"])]
len(grid_gdf)

In [ ]:
# ruff: noqa: N803, N806
def ordinary_kriging_simulation(
    u: gpd.GeoDataFrame,
    z: gpd.GeoSeries,
    geom: gpd.GeoSeries,
    model_fn: Callable,
    c0: float,
    c: float,
    a: float,
    filter_range: bool = True,
    neighbor_factor: float = 0.2,
) -> xr.DataArray:
    # Copy z and geom to avoid modifying the input
    z = z.copy()
    geom = geom.copy()

    # Shuffle unknown points to get a random path
    orig_index = u.index
    u = u.sample(frac=1)

    # Precaulate the pairwise distances and their respective covariance matrices by puttin together all geometries
    # In the loop we then can take slices of that large matrix
    # With this we avoid recalculation and concatenation of the covariance matrices in the loop
    with stopuhr("Pre-Caculating C"):
        all_geoms = pd.concat([geom, u.set_index(u.index + geom.index.max()).geometry])
        d_all = fastdist(all_geoms, all_geoms)
        d_all = d_all.rename({"a": "head", "b": "support"})
        neighbor_range = a * neighbor_factor
        neighbors = d_all <= neighbor_range
        C_all = cov_from_semivar(model_fn(d_all, c0, c, a), c0, c)
        C_all = C_all.where(d_all <= a, 0)

    # Turn z into an xarray DataArray with space for the unknown points
    z_all = xr.DataArray(0, dims=["support"], coords={"support": all_geoms.index}, name="z")
    z_all.loc[{"support": z.index}] = z

    # Create an empty mu array which will be filled
    mu = xr.DataArray(0, dims=["support"], coords={"support": u.index}, name="mu")

    dataset_indicator = xr.DataArray(
        False, dims=["support"], coords={"support": all_geoms.index}, name="dataset_indicator"
    )
    dataset_indicator.loc[{"support": geom.index}] = True

    # Move everything to cuda if available
    if cp is not None:
        dataset_indicator = dataset_indicator.cupy.as_cupy()
        z_all = z_all.cupy.as_cupy()
        mu = mu.cupy.as_cupy()

    # Create random generator
    rng = np.random.default_rng()

    # i: Iteration counter -> The number of already interpolated points
    # uidx: The index of the unknown point
    # ui: The unknown point
    for i, (uidx, ui) in track(enumerate(u.iterrows()), total=len(u)):
        # Take slices based on current iteration
        # Our C0 and Ch grows with each iteration, because we add more points to our dataset
        # C0 grows linear and Ch grows quadratically
        with stopuhr("Slicing C", res=4, loop=True):
            uidx_adj = uidx + geom.index.max()
            nc = (neighbors.sel(head=uidx_adj) & dataset_indicator).as_numpy().data
            C0 = C_all.sel(head=uidx_adj)[{"support": nc}]
            Ch = C_all[{"support": nc, "head": nc}]
        # C0 = C_all.sel(head=uidx_adj).isel(support=slice(0, len(geom) + i))
        # Ch = C_all.isel(head=slice(0, len(geom) + i)).isel(support=slice(0, len(geom) + i))

        # Apply the constraints that the lambdas must sum up to 1
        with stopuhr("Constraining C", res=4, loop=True):
            C0_constrained = C0.pad({"support": (0, 1)}, constant_values=1)
            Ch_constrained = Ch.pad({"support": (0, 1), "head": (0, 1)}, constant_values=1)
            Ch_constrained[{"support": -1, "head": -1}] = 0

        # Calculate the weights on cuda if available, else via numpy
        with stopuhr("Calculating lambda", res=4, loop=True):
            if cp is not None:
                C0_constrained = C0_constrained.cupy.as_cupy()
                Ch_constrained = Ch_constrained.cupy.as_cupy()
                neighbors = neighbors.cupy.as_cupy()
                lam = cp.linalg.solve(
                    Ch_constrained.data,
                    C0_constrained.data,
                )
            else:
                lam = np.linalg.solve(
                    Ch_constrained.data,
                    C0_constrained.data,
                )
            lam = xr.DataArray(
                lam,
                coords=C0_constrained.coords,
                name="lambda",
            )
            lam, mu_current = lam[:-1], lam[-1]
            mu.loc[{"support": uidx}] = mu_current

        with stopuhr("Calculating kriged", res=4, loop=True):
            # kriged_current = z_all.isel(support=slice(0, len(geom) + i)) @ lam
            kriged_current = z_all[{"support": nc}] @ lam
            r = rng.uniform()
            z_sim = (kriged_current > r).astype(int)
            z_all.loc[{"support": uidx_adj}] = z_sim
            dataset_indicator.loc[{"support": uidx_adj}] = True

        with stopuhr("Freeing memory", res=4, loop=True):
            if cp is not None:
                del lam, kriged_current, z_sim
                cp.get_default_memory_pool().free_all_blocks()

    # Reindex the adjusted index to the original index
    z_all.coords["support"] = z_all.coords["support"] - geom.index.max()
    kriged = z_all.sel(support=orig_index)
    mu = mu.sel(support=orig_index)

    if cp is not None:
        kriged = kriged.cupy.as_numpy()
        mu = mu.cupy.as_numpy()
        cp.get_default_memory_pool().free_all_blocks()

    return kriged, mu


stopuhr.reset()
with stopuhr("Simulation"):
    k = 10
    z = waterquality["As"]
    z_encoded = (z >= k).astype(int)
    kriged_grid, mu_grid = ordinary_kriging_simulation(
        grid_gdf, z_encoded, waterquality.geometry, spherical_fast, c0, c, a, neighbor_factor=1
    )
    grid_gdf[f"simulated_{k}"] = kriged_grid.to_numpy()
    grid_gdf[f"sim_mu_{k}"] = mu_grid.to_numpy()
stopuhr.results()

# Visualize the grid
vmax = np.percentile(waterquality["As"], 95)
m = waterquality.explore(column="As", cmap="PiYG_r", tiles="CartoDB positron", vmax=vmax)
grid_gdf.explore(m=m, column="simulated_10", marker_kwds={"radius": 3, "fill": True})
surveyarea.boundary.explore(m=m, style_kwds={"fill": False, "color": "black"})
m

In [ ]:
data_variance = (waterquality["As"] >= 10).astype(int).std()
# kriged_variance = grid_gdf["kriged_10"].std()
simulated_variance = grid_gdf["simulated_10"].std()
print(f"Data Variance: {data_variance:.2f}")
# print(f"Kriged Variance: {kriged_variance:.2f}")
print(f"Simulated Variance: {simulated_variance:.2f}")
assert data_variance > kriged_variance, "The kriged variance is larger than the data variance, which is not plausible."